In [1]:
# CNN thrombin binder classifier

# ==============================================================================
# Self-contained script for training a 1D CNN model on Morgan fingerprints
# to classify Thrombin binders from the ChEMBL database.
#
# The script performs:
# 1. Data acquisition via SQL queries to a ChEMBL SQLite database.
# 2. Featurization of SMILES strings into Morgan fingerprints.
# 3. Stratified splitting of data into train, validation, and test sets.
# 4. Definition and training of a 1D Convolutional Neural Network (CNN).
# 5. Evaluation of the trained model on the test set.
# 6. Saving the best model's weights for later use.
#
# To run this script, you must have the required libraries installed:
# - pandas, numpy, scikit-learn
# - torch, rdkit
# ==============================================================================

import os
import sqlite3
import random
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report)

warnings.filterwarnings("ignore")

try:
    from rdkit import Chem
    from rdkit.Chem import AllChem, DataStructs
except ImportError:
    print("RDKit not found. Please install it using `conda install -c conda-forge rdkit`.")
    exit()

# ------------------------------------------------------------------------------
# 1) CONFIGURATION
# ------------------------------------------------------------------------------
# IMPORTANT: Update these paths and parameters as needed for your environment.
DB_PATH = "C:\\Users\\Pawan\\Desktop\\quantum based rug discovery\\chembl_35.db"
SAVE_MODEL_PATH = "C:\\Users\\Pawan\\Desktop\\quantum based rug discovery\\models\\cnn_thrombin.pt"
TARGET_CHEMBL_ID = "CHEMBL204"     # Thrombin

STRONG_THRESH_NM = 100             # < 100 nM => binder (1)
WEAK_THRESH_NM = 5000              # > 5000 nM => non-binder (0)

FINGERPRINT_BITS = 2048
FINGERPRINT_RADIUS = 2

TEST_SIZE = 0.2
VAL_SIZE = 0.1                     # of the *train* split
BATCH_SIZE = 128
EPOCHS = 25
LR = 1e-3
EARLY_STOP_PATIENCE = 5
SEED = 42

# Set device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"Using device: {DEVICE}")

# ------------------------------------------------------------------------------
# 2) DATA RETRIEVAL AND PREPARATION
# ------------------------------------------------------------------------------
print("\n[INFO] Querying ChEMBL for training data...")
conn = sqlite3.connect(DB_PATH)

# SQL query for strong binders
q_strong = f"""
SELECT cs.canonical_smiles AS smiles FROM activities a
JOIN assays ass ON a.assay_id = ass.assay_id
JOIN target_dictionary td ON ass.tid = td.tid
JOIN compound_structures cs ON a.molregno = cs.molregno
WHERE a.standard_type IN ('IC50', 'Ki') AND a.standard_units = 'nM'
  AND a.standard_value < {STRONG_THRESH_NM} AND ass.assay_type = 'B'
  AND td.chembl_id = '{TARGET_CHEMBL_ID}' AND cs.canonical_smiles IS NOT NULL
"""
df_strong = pd.read_sql_query(q_strong, conn).drop_duplicates(subset=["smiles"])

# SQL query for weak binders
q_weak = f"""
SELECT cs.canonical_smiles AS smiles FROM activities a
JOIN assays ass ON a.assay_id = ass.assay_id
JOIN target_dictionary td ON ass.tid = td.tid
JOIN compound_structures cs ON a.molregno = cs.molregno
WHERE a.standard_type IN ('IC50', 'Ki') AND a.standard_units = 'nM'
  AND a.standard_value > {WEAK_THRESH_NM} AND ass.assay_type = 'B'
  AND td.chembl_id = '{TARGET_CHEMBL_ID}' AND cs.canonical_smiles IS NOT NULL
"""
df_weak = pd.read_sql_query(q_weak, conn).drop_duplicates(subset=["smiles"])
conn.close()

df_strong["label"] = 1
df_weak["label"] = 0
df_all = pd.concat([df_strong, df_weak], ignore_index=True)
df_all = df_all.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

print(f"Data summary (counts by class):\n{df_all['label'].value_counts()}")

# ------------------------------------------------------------------------------
# 3) FEATURIZATION AND DATA SPLITTING
# ------------------------------------------------------------------------------
def smiles_to_morgan_bits(smiles, n_bits=FINGERPRINT_BITS, radius=FINGERPRINT_RADIUS):
    """Converts a SMILES string to a Morgan fingerprint bit vector."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    arr = np.zeros((n_bits,), dtype=np.int8)
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

print("\n[INFO] Featurizing molecules into Morgan fingerprints...")
df_all["fp"] = df_all["smiles"].apply(smiles_to_morgan_bits)
df_all = df_all[df_all["fp"].notnull()].reset_index(drop=True)

X = np.stack(df_all["fp"].values)
y = df_all["label"].astype(int).values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=VAL_SIZE, random_state=SEED, stratify=y_train
)

print(f"Train set: {X_train.shape[0]} samples")
print(f"Validation set: {X_val.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

# ------------------------------------------------------------------------------
# 4) PYTORCH DATASET, MODEL, AND TRAINING
# ------------------------------------------------------------------------------
class FPSDataset(Dataset):
    """Custom PyTorch Dataset for fingerprint data."""
    def __init__(self, X, y):
        self.X = torch.from_numpy(X.astype(np.float32))
        self.y = torch.from_numpy(y.astype(np.int64))

    def __len__(self):
        return self.X.size(0)

    def __getitem__(self, idx):
        # Add a channel dimension for the 1D CNN
        return self.X[idx].unsqueeze(0), self.y[idx]

train_ds = FPSDataset(X_train, y_train)
val_ds = FPSDataset(X_val, y_val)
test_ds = FPSDataset(X_test, y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

class_counts = np.bincount(y_train)
pos_weight = class_counts[0] / (class_counts[1] + 1e-6)

class CNN1D(nn.Module):
    """A simple 1D CNN for classifying Morgan fingerprints."""
    def __init__(self, in_len=FINGERPRINT_BITS, n_classes=1):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 64, kernel_size=7, padding=3)
        self.bn1 = nn.BatchNorm1d(64)
        self.conv2 = nn.Conv1d(64, 128, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(128)
        self.conv3 = nn.Conv1d(128, 256, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm1d(256)
        self.drop = nn.Dropout(0.3)
        self.clf = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, n_classes)
        )

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.max_pool1d(x, 2)
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.max_pool1d(x, 2)
        x = F.relu(self.bn3(self.conv3(x)))
        x = F.adaptive_max_pool1d(x, 1).squeeze(-1)
        x = self.drop(x)
        return self.clf(x).squeeze(-1)

model = CNN1D().to(DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight], device=DEVICE))
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

def run_epoch(loader, train=True):
    """Runs a single training or validation epoch."""
    model.train(train)
    total_loss, n = 0.0, 0
    all_logits, all_targets = [], []
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.float().to(DEVICE)
        logits = model(xb)
        loss = criterion(logits, yb)
        if train:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        total_loss += loss.item() * xb.size(0)
        n += xb.size(0)
        all_logits.append(logits.detach().cpu())
        all_targets.append(yb.detach().cpu())
    
    avg_loss = total_loss / max(n, 1)
    logits = torch.cat(all_logits).numpy()
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)
    targets = torch.cat(all_targets).numpy().astype(int)
    acc = accuracy_score(targets, preds)
    try:
        auc = roc_auc_score(targets, probs)
    except ValueError:
        auc = float("nan")
    return avg_loss, acc, auc

best_val_loss = float("inf")
best_state = None
epochs_no_improve = 0

print("\n[INFO] Starting model training...")
for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc, tr_auc = run_epoch(train_loader, train=True)
    val_loss, val_acc, val_auc = run_epoch(val_loader, train=False)
    scheduler.step(val_loss)
    
    print(f"Epoch {epoch:02d} | Train: loss={tr_loss:.4f} acc={tr_acc:.3f} auc={tr_auc:.3f} | "
          f"Val: loss={val_loss:.4f} acc={val_acc:.3f} auc={val_auc:.3f}")
    
    if val_loss < best_val_loss - 1e-4:
        best_val_loss = val_loss
        best_state = {k: v.cpu() for k, v in model.state_dict().items()}
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= EARLY_STOP_PATIENCE:
            print("[INFO] Early stopping triggered.")
            break

if best_state is not None:
    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})

# ------------------------------------------------------------------------------
# 5) FINAL EVALUATION ON TEST SET
# ------------------------------------------------------------------------------
model.eval()
all_probs, all_preds, all_true = [], [], []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(DEVICE)
        yb = yb.cpu().numpy().astype(int)
        logits = model(xb).cpu().numpy()
        probs = 1 / (1 + np.exp(-logits))
        preds = (probs >= 0.5).astype(int)
        
        all_probs.extend(probs.tolist())
        all_preds.extend(preds.tolist())
        all_true.extend(yb.tolist())

acc = accuracy_score(all_true, all_preds)
prec = precision_score(all_true, all_preds, zero_division=0)
rec = recall_score(all_true, all_preds, zero_division=0)
f1 = f1_score(all_true, all_preds, zero_division=0)
try:
    auc = roc_auc_score(all_true, all_probs)
except ValueError:
    auc = float("nan")

print("\n=== Test Set Metrics ===")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-score : {f1:.4f}")
print(f"ROC-AUC  : {auc:.4f}")

cm = confusion_matrix(all_true, all_preds)
print("\nConfusion Matrix (rows=true, cols=pred):\n", cm)
print("\nClassification Report:\n", classification_report(all_true, all_preds, target_names=['Non-binder','Binder']))

# ------------------------------------------------------------------------------
# 6) SAVE MODEL WEIGHTS
# ------------------------------------------------------------------------------
try:
    ensure_dir(os.path.dirname(SAVE_MODEL_PATH) or ".")
    torch.save(model.state_dict(), SAVE_MODEL_PATH)
    print(f"\n[SUCCESS] Trained model saved to: {SAVE_MODEL_PATH}")
except Exception as e:
    print(f"\n[ERROR] Failed to save the model. Reason: {e}")


Using device: cpu

[INFO] Querying ChEMBL for training data...
Data summary (counts by class):
label
0    2300
1    1968
Name: count, dtype: int64

[INFO] Featurizing molecules into Morgan fingerprints...
Train set: 3072 samples
Validation set: 342 samples
Test set: 854 samples

[INFO] Starting model training...
Epoch 01 | Train: loss=0.7816 acc=0.532 auc=0.548 | Val: loss=0.7436 acc=0.538 auc=0.704
Epoch 02 | Train: loss=0.7139 acc=0.595 auc=0.637 | Val: loss=0.7026 acc=0.623 auc=0.724
Epoch 03 | Train: loss=0.6711 acc=0.649 auc=0.710 | Val: loss=0.6373 acc=0.708 auc=0.774
Epoch 04 | Train: loss=0.6727 acc=0.651 auc=0.708 | Val: loss=0.6579 acc=0.673 auc=0.795
Epoch 05 | Train: loss=0.6302 acc=0.695 auc=0.761 | Val: loss=0.6059 acc=0.728 auc=0.815
Epoch 06 | Train: loss=0.5966 acc=0.715 auc=0.790 | Val: loss=0.5775 acc=0.760 auc=0.835
Epoch 07 | Train: loss=0.5967 acc=0.719 auc=0.791 | Val: loss=0.5822 acc=0.740 auc=0.840
Epoch 08 | Train: loss=0.5743 acc=0.738 auc=0.811 | Val: loss=0

In [ ]:
# =========================
# Thrombin binder classifier with 1D CNN on Morgan fingerprints
# =========================
import os, sqlite3, math, random, warnings
warnings.filterwarnings("ignore")

# -------------------------
# 1) Config
# -------------------------
DB_PATH = r"C:\Users\Pawan\Desktop\quantum based rug discovery\chembl_35.db"
TARGET_CHEMBL_ID = "CHEMBL204"         # Thrombin
STRONG_THRESH_NM = 100                 # < 100 nM => binder (1)
WEAK_THRESH_NM   = 5000                # > 5000 nM => non-binder (0)
N_STRONG_LIMIT   = None                # set e.g. 5000 to cap, or None for all
N_WEAK_LIMIT     = 5000                # cap to limit extreme imbalance
FINGERPRINT_BITS = 2048
FINGERPRINT_RADIUS = 2
TEST_SIZE = 0.2
VAL_SIZE  = 0.1                         # of the *train* split
BATCH_SIZE = 128
EPOCHS = 25
LR = 1e-3
EARLY_STOP_PATIENCE = 5
SEED = 42

random.seed(SEED)
import numpy as np
np.random.seed(SEED)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report)
import matplotlib.pyplot as plt

print(f"Using DEVICE={DEVICE}")
try:
    import rdkit
    print("NumPy:", np.__version__)
    print("RDKit:", rdkit.__version__)
except Exception as e:
    print("Version check note:", e)

# -------------------------
# 2) Read data from ChEMBL
# -------------------------
conn = sqlite3.connect(DB_PATH)

# Strong binders: < STRONG_THRESH_NM
q_strong = f"""
SELECT
    md.chembl_id AS molecule_chembl_id,
    cs.canonical_smiles AS smiles,
    a.standard_type,
    a.standard_value,
    a.standard_units,
    td.chembl_id AS target_chembl_id,
    td.pref_name  AS target_name
FROM activities a
JOIN assays ass ON a.assay_id = ass.assay_id
JOIN target_dictionary td ON ass.tid = td.tid
JOIN molecule_dictionary md ON a.molregno = md.molregno
JOIN compound_structures cs ON a.molregno = cs.molregno
WHERE a.standard_type IN ('IC50','Ki')
  AND a.standard_units = 'nM'
  AND a.standard_value < {STRONG_THRESH_NM}
  AND ass.assay_type = 'B'
  AND td.chembl_id = '{TARGET_CHEMBL_ID}'
  AND cs.canonical_smiles IS NOT NULL
"""
if N_STRONG_LIMIT:
    q_strong += f"\nLIMIT {int(N_STRONG_LIMIT)}"

# Weak binders: > WEAK_THRESH_NM
q_weak = f"""
SELECT
    md.chembl_id AS molecule_chembl_id,
    cs.canonical_smiles AS smiles,
    a.standard_type,
    a.standard_value,
    a.standard_units,
    td.chembl_id AS target_chembl_id,
    td.pref_name  AS target_name
FROM activities a
JOIN assays ass ON a.assay_id = ass.assay_id
JOIN target_dictionary td ON ass.tid = td.tid
JOIN molecule_dictionary md ON a.molregno = md.molregno
JOIN compound_structures cs ON a.molregno = cs.molregno
WHERE a.standard_type IN ('IC50','Ki')
  AND a.standard_units = 'nM'
  AND a.standard_value > {WEAK_THRESH_NM}
  AND ass.assay_type = 'B'
  AND td.chembl_id = '{TARGET_CHEMBL_ID}'
  AND cs.canonical_smiles IS NOT NULL
"""
if N_WEAK_LIMIT:
    q_weak += f"\nLIMIT {int(N_WEAK_LIMIT)}"

df_strong = pd.read_sql_query(q_strong, conn)
df_weak   = pd.read_sql_query(q_weak,   conn)
conn.close()

df_strong["label"] = 1
df_weak["label"]   = 0

df_all = pd.concat([df_strong, df_weak], ignore_index=True)
df_all.drop_duplicates(subset=["smiles","standard_type","standard_value"], inplace=True)
df_all = df_all.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

print("Data summary (counts by class):")
print(df_all.groupby("label")["smiles"].count())
print(df_all.head())

# -------------------------
# 3) Featurization: Morgan FP
# -------------------------
def smiles_to_morgan_bits(smiles, n_bits=FINGERPRINT_BITS, radius=FINGERPRINT_RADIUS):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    arr = np.zeros((n_bits,), dtype=np.int8)
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

df_all["fp"] = df_all["smiles"].apply(smiles_to_morgan_bits)
df_all = df_all[df_all["fp"].notnull()].reset_index(drop=True)

X = np.stack(df_all["fp"].values)  # (N, 2048)
y = df_all["label"].astype(int).values
idx_all = np.arange(len(df_all))   # keep mapping to original rows

# -------------------------
# 4) Train/val/test split (stratified) — keep indices for SMILES mapping
# -------------------------
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, idx_all, test_size=TEST_SIZE, random_state=SEED, stratify=y
)
X_train, X_val, y_train, y_val, idx_train, idx_val = train_test_split(
    X_train, y_train, idx_train, test_size=VAL_SIZE, random_state=SEED, stratify=y_train
)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

# -------------------------
# Save processed dataset (without the 'fp' column for a clean CSV)
# -------------------------
df_all_no_fp = df_all.drop(columns=["fp"])
df_all_no_fp.to_csv("thrombin_dataset.csv", index=False)
print("Processed dataset saved to thrombin_dataset.csv")

# -------------------------
# 5) PyTorch Dataset/Dataloader (return original indices too)
# -------------------------
class FPSDataset(Dataset):
    def __init__(self, X, y, idx):
        self.X = torch.from_numpy(X.astype(np.float32))  # shape (N, 2048)
        self.y = torch.from_numpy(y.astype(np.int64))
        self.idx = np.array(idx, dtype=int)

    def __len__(self):
        return self.X.size(0)

    def __getitem__(self, i):
        # For 1D CNN we need shape (C=1, L=2048)
        x = self.X[i].unsqueeze(0)
        return x, self.y[i], int(self.idx[i])

train_ds = FPSDataset(X_train, y_train, idx_train)
val_ds   = FPSDataset(X_val,   y_val,   idx_val)
test_ds  = FPSDataset(X_test,  y_test,  idx_test)

# Optional: class imbalance handling via pos_weight
class_counts = np.bincount(y_train)
pos_weight = class_counts[0] / (class_counts[1] + 1e-6)  # for BCEWithLogitsLoss
print(f"Train class counts: {class_counts}, pos_weight (for class 1) ~ {pos_weight:.3f}")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  drop_last=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

# -------------------------
# 6) 1D CNN model
# -------------------------
class CNN1D(nn.Module):
    def __init__(self, in_len=FINGERPRINT_BITS, n_classes=1):
        super().__init__()
        # Input: (B, 1, 2048)
        self.conv1 = nn.Conv1d(1, 64, kernel_size=7, padding=3)
        self.bn1   = nn.BatchNorm1d(64)
        self.conv2 = nn.Conv1d(64, 128, kernel_size=5, padding=2)
        self.bn2   = nn.BatchNorm1d(128)
        self.conv3 = nn.Conv1d(128, 256, kernel_size=3, padding=1)
        self.bn3   = nn.BatchNorm1d(256)
        self.dropout = nn.Dropout(0.3)

        # Global max pool over sequence length
        self.classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, n_classes)  # logits
        )

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.max_pool1d(x, kernel_size=2)   # (B, 64, 1024)
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.max_pool1d(x, kernel_size=2)   # (B, 128, 512)
        x = F.relu(self.bn3(self.conv3(x)))
        x = F.adaptive_max_pool1d(x, output_size=1).squeeze(-1) # (B, 256)
        x = self.dropout(x)
        logits = self.classifier(x).squeeze(-1)
        return logits

model = CNN1D().to(DEVICE)

# Loss/optimizer/scheduler (no 'verbose' arg for compatibility)
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight], device=DEVICE))
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

def make_plateau_scheduler(optim):
    kwargs = dict(mode="min", factor=0.5, patience=2, threshold=1e-4, cooldown=0, min_lr=1e-6)
    try:
        return torch.optim.lr_scheduler.ReduceLROnPlateau(optim, verbose=True, **kwargs)
    except TypeError:
        return torch.optim.lr_scheduler.ReduceLROnPlateau(optim, **kwargs)

scheduler = make_plateau_scheduler(optimizer)

# -------------------------
# 7) Training loop with early stopping
# -------------------------
def run_epoch(loader, train=True):
    model.train(train)
    total_loss, n = 0.0, 0
    all_logits, all_targets = [], []
    for xb, yb, _idx in loader:
        xb = xb.to(DEVICE)
        yb = yb.float().to(DEVICE)   # BCE expects float targets

        logits = model(xb)
        loss = criterion(logits, yb)

        if train:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * xb.size(0)
        n += xb.size(0)
        all_logits.append(logits.detach().cpu())
        all_targets.append(yb.detach().cpu())

    avg_loss = total_loss / max(n, 1)
    logits = torch.cat(all_logits).numpy()
    probs = 1/(1+np.exp(-logits))
    preds = (probs >= 0.5).astype(int)
    targets = torch.cat(all_targets).numpy().astype(int)
    acc  = accuracy_score(targets, preds)
    try:
        auc  = roc_auc_score(targets, probs)
    except ValueError:
        auc = float("nan")
    return avg_loss, acc, auc

best_val_loss = float("inf")
best_state = None
epochs_no_improve = 0

for epoch in range(1, EPOCHS+1):
    tr_loss, tr_acc, tr_auc = run_epoch(train_loader, train=True)
    val_loss, val_acc, val_auc = run_epoch(val_loader,   train=False)

    prev_lr = optimizer.param_groups[0]["lr"]
    scheduler.step(val_loss)
    new_lr = optimizer.param_groups[0]["lr"]
    if new_lr < prev_lr:
        print(f"LR reduced: {prev_lr:.2e} → {new_lr:.2e}")

    print(f"Epoch {epoch:02d} | "
          f"Train: loss={tr_loss:.4f} acc={tr_acc:.3f} auc={tr_auc:.3f} | "
          f"Val: loss={val_loss:.4f} acc={val_acc:.3f} auc={val_auc:.3f}")

    if val_loss < best_val_loss - 1e-4:
        best_val_loss = val_loss
        best_state = {k: v.cpu() for k, v in model.state_dict().items()}
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= EARLY_STOP_PATIENCE:
            print("Early stopping.")
            break

if best_state is not None:
    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})

# -------------------------
# 8) Evaluation on test set
# -------------------------
model.eval()
all_probs, all_preds, all_true, all_idx = [], [], [], []
with torch.no_grad():
    for xb, yb, ib in test_loader:
        xb = xb.to(DEVICE)
        logits = model(xb).cpu().numpy()
        probs = 1/(1+np.exp(-logits))
        preds = (probs >= 0.5).astype(int)

        all_probs.extend(probs.tolist())
        all_preds.extend(preds.tolist())
        all_true.extend(yb.numpy().astype(int).tolist())
        all_idx.extend(ib.numpy().astype(int).tolist())

acc  = accuracy_score(all_true, all_preds)
prec = precision_score(all_true, all_preds, zero_division=0)
rec  = recall_score(all_true, all_preds, zero_division=0)
f1   = f1_score(all_true, all_preds, zero_division=0)
try:
    auc  = roc_auc_score(all_true, all_probs)
except ValueError:
    auc = float("nan")

print("\n=== Test Metrics ===")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-score : {f1:.4f}")
print(f"ROC-AUC  : {auc:.4f}")

cm = confusion_matrix(all_true, all_preds)
print("\nConfusion Matrix (rows=true, cols=pred):\n", cm)
print("\nClassification Report:")
print(classification_report(all_true, all_preds, target_names=['Non-binder','Binder']))

# -------------------------
# 9) (Optional) Quick diagnostic plots
# -------------------------
plt.figure()
plt.hist(np.array(all_probs)[np.array(all_true)==0], bins=50, alpha=0.6, label="Non-binder")
plt.hist(np.array(all_probs)[np.array(all_true)==1], bins=50, alpha=0.6, label="Binder")
plt.xlabel("Predicted probability (Binder)")
plt.ylabel("Count")
plt.title("Score Distribution")
plt.legend()
plt.tight_layout()
plt.show()

# -------------------------
# 10) Save test predictions (+ SMILES)
# -------------------------
# Map test indices (into df_all rows) to SMILES
smiles_test = df_all.loc[all_idx, "smiles"].tolist()
mol_chembl_ids_test = df_all.loc[all_idx, "molecule_chembl_id"].tolist()

results = pd.DataFrame({
    "molecule_chembl_id": mol_chembl_ids_test,
    "smiles": smiles_test,
    "true_label": all_true,
    "predicted_label": all_preds,
    "predicted_prob": all_probs
})
results.to_csv("cnn_predictions.csv", index=False)
print("Test predictions saved to cnn_predictions.csv")


Using DEVICE=cpu
NumPy: 2.3.1
RDKit: 2024.03.6
Data summary (counts by class):
label
0    2381
1    2201
Name: smiles, dtype: int64
  molecule_chembl_id                                             smiles  \
0       CHEMBL428116  CC[C@@H](C)[C@H](NC(=O)[C@@H]1CCCN1C(=O)[C@H](...   
1        CHEMBL55500  N=C(N)c1ccc(CC(=O)C(=O)O)c(OCCNC(=O)C2CCN(c3cc...   
2      CHEMBL1811998  COC(=O)c1c(C)oc2cc(OC)c(OS(=O)(=O)[O-])cc12.[Na+]   
3      CHEMBL2159378         COc1ccc(CNS(=O)(=O)c2cccc(C(=N)N)c2)cc1.Cl   
4      CHEMBL2315237  N=C(N)c1ccc(CNC(=O)[C@@H]2Cc3ccc(cc3)NC(=O)CN3...   

  standard_type  standard_value standard_units target_chembl_id target_name  \
0            Ki    5.700000e-04             nM        CHEMBL204    Thrombin   
1            Ki    1.400000e+05             nM        CHEMBL204    Thrombin   
2          IC50    3.000000e+06             nM        CHEMBL204    Thrombin   
3            Ki    2.450000e+05             nM        CHEMBL204    Thrombin   
4            Ki    5.8

In [ ]:
# -------------------------
# 10) Save test predictions (+ SMILES)
# -------------------------
# Map test indices (into df_all rows) to SMILES
smiles_test = df_all.loc[all_idx, "smiles"].tolist()
mol_chembl_ids_test = df_all.loc[all_idx, "molecule_chembl_id"].tolist()

results = pd.DataFrame({
    "molecule_chembl_id": mol_chembl_ids_test,
    "smiles": smiles_test,
    "true_label": all_true,
    "predicted_label": all_preds,
    "predicted_prob": all_probs
})
results.to_csv("cnn_predictions.csv", index=False)
print("Test predictions saved to cnn_predictions.csv")

# -------------------------
# 11) Save trained model weights
# -------------------------
torch.save(model.state_dict(), "CNN_thrombin.pt")
print("Trained model weights saved to CNN_thrombin.pt")


In [ ]:
import os

OUT_DIR = r"C:\Users\Pawan\Desktop\quantum based rug discovery"
os.makedirs(OUT_DIR, exist_ok=True)

# Add the label you already computed and keep core fields
df_out = df_all.loc[:, [
    "molecule_chembl_id", "smiles", "standard_type", "standard_value",
    "standard_units", "target_chembl_id", "target_name", "label"
]].copy()

csv_path = os.path.join(OUT_DIR, "ligand_binding_data.csv")
df_out.to_csv(csv_path, index=False, encoding="utf-8")
print("Wrote:", csv_path)


In [ ]:
# attach predictions for all rows, not just test
with torch.no_grad():
    X_all = torch.from_numpy(np.stack(df_all["fp"].values).astype(np.float32)).unsqueeze(1).to(DEVICE)
    logits_all = model(X_all).cpu().numpy()
    probs_all = 1/(1+np.exp(-logits_all))

df_all["cnn_prob"] = probs_all
df_all.loc[:, [
    "molecule_chembl_id","smiles","standard_type","standard_value","standard_units",
    "target_chembl_id","target_name","label","cnn_prob"
]].to_csv(os.path.join(OUT_DIR, "ligand_binding_data.csv"), index=False, encoding="utf-8")
print("Wrote (with cnn_prob):", os.path.join(OUT_DIR, "ligand_binding_data.csv"))


In [ ]:
import pandas as pd
import numpy as np
import torch, math, os
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs

CSV_IN  = r"C:\Users\Pawan\Desktop\quantum based rug discovery\ligand_binding_data.csv"
MODEL   = r"C:\Users\Pawan\Desktop\quantum based rug discovery\models\cnn_thrombin.pt"
CSV_OUT = r"C:\Users\Pawan\Desktop\quantum based rug discovery\ligands_labeled_scored.csv"

STRONG_THRESH_NM = 100
WEAK_THRESH_NM   = 5000
BITS = 2048
RADIUS = 2
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- load and normalize headers ---
df = pd.read_csv(CSV_IN)
df.columns = df.columns.str.strip().str.lower()  # remove spaces, lowercase

# Expected names after normalization:
# 'molecule_chembl_id','smiles','standard_type','standard_value','standard_units','target_chembl_id','target_name'

# pick the smiles column robustly
smiles_col = None
for cand in ["canonical_smiles", "smiles"]:
    if cand in df.columns:
        smiles_col = cand
        break
if smiles_col is None:
    raise KeyError(f"No SMILES column found. Got columns: {list(df.columns)}")

# force standard_value to numeric (coerce weird strings)
df["standard_value"] = pd.to_numeric(df["standard_value"], errors="coerce")

# --- add labels from nM thresholds ---
def assign_label(nm):
    if pd.isna(nm): 
        return np.nan
    # smaller nM => stronger binder
    if nm < STRONG_THRESH_NM:
        return 1
    if nm > WEAK_THRESH_NM:
        return 0
    return np.nan  # grey zone ignored

df["label"] = df["standard_value"].apply(assign_label)
df = df[df["label"].notna()].copy()

# --- CNN model def (must match training) ---
import torch.nn as nn
import torch.nn.functional as F

class CNN1D(nn.Module):
    def __init__(self, in_len=2048, n_classes=1):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 64, kernel_size=7, padding=3); self.bn1 = nn.BatchNorm1d(64)
        self.conv2 = nn.Conv1d(64,128, kernel_size=5, padding=2); self.bn2 = nn.BatchNorm1d(128)
        self.conv3 = nn.Conv1d(128,256, kernel_size=3, padding=1); self.bn3 = nn.BatchNorm1d(256)
        self.drop  = nn.Dropout(0.3)
        self.clf   = nn.Sequential(nn.Linear(256,128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128,1))
    def forward(self,x):
        x = F.relu(self.bn1(self.conv1(x))); x = F.max_pool1d(x,2)
        x = F.relu(self.bn2(self.conv2(x))); x = F.max_pool1d(x,2)
        x = F.relu(self.bn3(self.conv3(x))); x = F.adaptive_max_pool1d(x,1).squeeze(-1)
        x = self.drop(x); return self.clf(x).squeeze(-1)

def smiles_to_bits(smiles, n_bits=BITS, radius=RADIUS):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    arr = np.zeros((n_bits,), dtype=np.int8)
    fp  = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

# load trained weights
model = CNN1D(in_len=BITS).to(DEVICE)
model.load_state_dict(torch.load(MODEL, map_location=DEVICE))
model.eval()

def predict_prob(smiles):
    bits = smiles_to_bits(smiles)
    if bits is None: return np.nan
    t = torch.from_numpy(bits.astype(np.float32)).unsqueeze(0).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logit = model(t).item()
    return 1.0 / (1.0 + math.exp(-logit))

# compute cnn_prob
df["cnn_prob"] = df[smiles_col].apply(predict_prob)

# standardize final column names for downstream
rename_map = {
    smiles_col: "smiles",
    "molecule_chembl_id": "name"
}
df.rename(columns={k:v for k,v in rename_map.items() if k in df.columns}, inplace=True)

# keep only what later steps need
out_cols = ["name","smiles","label","standard_value","cnn_prob"]
# include name if present; otherwise drop it from list
out_cols = [c for c in out_cols if c in df.columns]
df[out_cols].to_csv(CSV_OUT, index=False)
print("Saved:", CSV_OUT)


In [ ]:
# import pandas as pd
# import numpy as np
# import torch, math, os
# from rdkit import Chem
# from rdkit.Chem import AllChem, DataStructs

# # --- paths ---
# CSV_IN  = r"C:\Users\Pawan\Desktop\quantum based rug discovery\ligand_binding_data.csv"
# MODEL   = r"C:\Users\Pawan\Desktop\quantum based rug discovery\models\cnn_thrombin.pt"
# CSV_OUT = r"C:\Users\Pawan\Desktop\quantum based rug discovery\ligands_labeled_scored.csv"

# STRONG_THRESH_NM = 100
# WEAK_THRESH_NM   = 5000
# BITS = 2048
# RADIUS = 2
# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# df = pd.read_csv(CSV_IN)

# # 0.1 Add labels
# def assign_label(nm):
#     if pd.isna(nm): 
#         return np.nan
#     if nm < STRONG_THRESH_NM: 
#         return 1
#     if nm > WEAK_THRESH_NM:   
#         return 0
#     return np.nan  # grey zone

# df["label"] = df["standard_value"].apply(assign_label)
# df = df[df["label"].notna()].copy()  # keep only strong/weak bins

# # 0.2 Compute cnn_prob with your trained model
# import torch.nn as nn
# import torch.nn.functional as F

# class CNN1D(nn.Module):
#     def __init__(self, in_len=2048, n_classes=1):
#         super().__init__()
#         self.conv1 = nn.Conv1d(1, 64, kernel_size=7, padding=3)
#         self.bn1   = nn.BatchNorm1d(64)
#         self.conv2 = nn.Conv1d(64,128, kernel_size=5, padding=2)
#         self.bn2   = nn.BatchNorm1d(128)
#         self.conv3 = nn.Conv1d(128,256, kernel_size=3, padding=1)
#         self.bn3   = nn.BatchNorm1d(256)
#         self.drop  = nn.Dropout(0.3)
#         self.clf   = nn.Sequential(
#             nn.Linear(256,128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128,1)
#         )
#     def forward(self,x):
#         x = F.relu(self.bn1(self.conv1(x))); x = F.max_pool1d(x,2)
#         x = F.relu(self.bn2(self.conv2(x))); x = F.max_pool1d(x,2)
#         x = F.relu(self.bn3(self.conv3(x)))
#         x = F.adaptive_max_pool1d(x,1).squeeze(-1)
#         x = self.drop(x)
#         return self.clf(x).squeeze(-1)

# def smiles_to_bits(smiles, n_bits=BITS, radius=RADIUS):
#     mol = Chem.MolFromSmiles(smiles)
#     if mol is None: return None
#     arr = np.zeros((n_bits,), dtype=np.int8)
#     fp  = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
#     DataStructs.ConvertToNumpyArray(fp, arr)
#     return arr

# model = CNN1D(in_len=BITS).to(DEVICE)
# model.load_state_dict(torch.load(MODEL, map_location=DEVICE))
# model.eval()

# def predict_prob(smiles):
#     bits = smiles_to_bits(smiles)
#     if bits is None: return np.nan
#     t = torch.from_numpy(bits.astype(np.float32)).unsqueeze(0).unsqueeze(0).to(DEVICE)
#     with torch.no_grad():
#         logit = model(t).item()
#     return 1.0 / (1.0 + math.exp(-logit))

# df["cnn_prob"] = df["canonical_smiles"].apply(predict_prob)

# # 0.3 Save for downstream steps
# df.rename(columns={"canonical_smiles":"smiles", "molecule_chembl_id":"name"}, inplace=True)
# df[["name","smiles","label","standard_value","cnn_prob"]].to_csv(CSV_OUT, index=False)
# print("Saved:", CSV_OUT)


In [ ]:
# Step 5 — Prepare ligands and receptor (driven by labels + probs)

# Selection strategy: we want both high-probability binders and some non-binders for fair evaluation.

In [ ]:
import os, shutil, subprocess
import pandas as pd
from pathlib import Path

RUN_DIR   = r"C:\Users\Pawan\Desktop\quantum based rug discovery\runs\vs1"
LCSV      = r"C:\Users\Pawan\Desktop\quantum based rug discovery\ligands_labeled_scored.csv"
RECEPTOR  = r"C:\Users\Pawan\Desktop\quantum based rug discovery\receptor.pdb"  # full protein
POCKET    = r"C:\Users\Pawan\Desktop\quantum based rug discovery\pocket.pdb"    # pocket atoms from fpocket

PROB_THRESH = 0.5
TOP_K       = 100

def ensure_dir(d): Path(d).mkdir(parents=True, exist_ok=True)
ensure_dir(RUN_DIR)
prep_dir = os.path.join(RUN_DIR,"ligands_prep"); ensure_dir(prep_dir)

obabel = shutil.which("obabel")
if not obabel: raise RuntimeError("OpenBabel (obabel) not found on PATH")

df = pd.read_csv(LCSV).dropna(subset=["cnn_prob","label"]).copy()
df.sort_values("cnn_prob", ascending=False, inplace=True)

# Balance classes: top-K/2 predicted binders and top-K/2 non-binders
half = TOP_K // 2
pos = df[df["cnn_prob"] >= PROB_THRESH]
dock_pos = pos.head(half)
dock_neg = df[df["label"]==0].head(half)
dock_set = pd.concat([dock_pos, dock_neg]).drop_duplicates("name").head(TOP_K).copy()

# Save screened set
dock_set.to_csv(os.path.join(RUN_DIR,"screened_ligands.csv"), index=False)

# Receptor to PDBQT
receptor_pdbqt = os.path.join(RUN_DIR,"receptor.pdbqt")
subprocess.run([obabel, RECEPTOR, "-O", receptor_pdbqt, "-h", "--partialcharge", "gasteiger"], check=True)

# Ligands: SMILES -> SDF (RDKit) -> PDBQT (obabel)
from rdkit.Chem import AllChem

def smiles_to_sdf(smiles, out_sdf, seed=42):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return False
    mol = Chem.AddHs(mol)
    params = AllChem.ETKDGv3(); params.randomSeed = seed
    if AllChem.EmbedMolecule(mol, params) != 0:
        return False
    try: AllChem.UFFOptimizeMolecule(mol, maxIters=500)
    except: pass
    w = Chem.SDWriter(out_sdf); w.write(mol); w.close()
    return True

prep_records = []
for _, r in dock_set.iterrows():
    name, smi = r["name"], r["smiles"]
    sdf = os.path.join(prep_dir, f"{name}.sdf")
    pdbqt = os.path.join(prep_dir, f"{name}.pdbqt")
    ok = smiles_to_sdf(smi, sdf)
    if not ok: 
        prep_records.append((name, False, "SDF gen failed")); 
        continue
    res = subprocess.run([obabel, sdf, "-O", pdbqt, "-h", "--partialcharge", "gasteiger"],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    prep_records.append((name, os.path.exists(pdbqt), "ok" if os.path.exists(pdbqt) else "obabel fail"))

pd.DataFrame(prep_records, columns=["name","ok","note"]).to_csv(os.path.join(RUN_DIR,"prep_status.csv"), index=False)
print("Prep done.")


In [ ]:
# Docking box (from pocket atoms):

In [ ]:
def pdb_center_from_atoms(pdb_path):
    xs, ys, zs = [], [], []
    with open(pdb_path) as f:
        for line in f:
            if line.startswith(("ATOM","HETATM")):
                try:
                    xs.append(float(line[30:38])); ys.append(float(line[38:46])); zs.append(float(line[46:54]))
                except: pass
    if not xs: raise ValueError("No atom coords found for pocket.")
    return (sum(xs)/len(xs), sum(ys)/len(ys), sum(zs)/len(zs))

center = pdb_center_from_atoms(POCKET)
box = (20.0, 20.0, 20.0)  # safe default; tune using fpocket radius if desired
print("Dock center:", center, "box:", box)


In [ ]:
# Step 6 — Docking + metrics that use your labels

In [ ]:
import numpy as np, subprocess, os, pandas as pd
from pathlib import Path

vina = shutil.which("vina")  # or gnina
if not vina: raise RuntimeError("vina not found on PATH")

pose_dir = os.path.join(RUN_DIR,"poses"); ensure_dir(pose_dir)
screened = pd.read_csv(os.path.join(RUN_DIR,"screened_ligands.csv"))

def parse_vina_best(log_text):
    # Parse the first "mode" line: "   1       -7.8 ..."
    for line in (log_text or "").splitlines():
        parts = line.strip().split()
        if len(parts)>=2 and parts[0].isdigit():
            try: return float(parts[1])
            except: pass
    return None

rows = []
for _, r in screened.iterrows():
    name = r["name"]
    lig_pdbqt = os.path.join(RUN_DIR, "ligands_prep", f"{name}.pdbqt")
    out_pose  = os.path.join(pose_dir, f"{name}_pose.pdbqt")
    if not os.path.exists(lig_pdbqt): 
        continue
    cmd = [
        vina,
        "--receptor", os.path.join(RUN_DIR,"receptor.pdbqt"),
        "--ligand",   lig_pdbqt,
        "--center_x", str(center[0]), "--center_y", str(center[1]), "--center_z", str(center[2]),
        "--size_x", str(box[0]), "--size_y", str(box[1]), "--size_z", str(box[2]),
        "--exhaustiveness", "16",
        "--out", out_pose
    ]
    p = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    aff = parse_vina_best(p.stdout)
    rows.append({
        "name": name,
        "smiles": r["smiles"],
        "label": int(r["label"]),
        "cnn_prob": float(r["cnn_prob"]),
        "dock_affinity_kcalmol": aff,
        "pose_path": out_pose if os.path.exists(out_pose) else ""
    })

res = pd.DataFrame(rows).dropna(subset=["dock_affinity_kcalmol"])
res.to_csv(os.path.join(RUN_DIR,"docking_results.csv"), index=False)
print("Docking done:", len(res), "poses")


In [ ]:
# Evaluate enrichment & correlation:

In [ ]:
from sklearn.metrics import roc_auc_score
import numpy as np

df = res.copy()
# Prospective ROC-AUC of cnn_prob vs label (on docked set)
if df["label"].nunique()==2:
    auc = roc_auc_score(df["label"], df["cnn_prob"])
    print(f"[METRIC] Prospective AUC (cnn_prob vs label): {auc:.3f}")

# Enrichment Factor @k
def EF_at_k(df, k):
    k = min(k, len(df))
    topk = df.sort_values("cnn_prob", ascending=False).head(k)
    actives_in_topk = topk["label"].sum()
    base_rate = df["label"].mean()
    return (actives_in_topk / k) / base_rate if base_rate>0 else np.nan

for k in [10, 25, 50]:
    print(f"[METRIC] EF@{k}: {EF_at_k(df, k):.2f}")

# Correlation (cnn_prob vs docking; more negative docking is better)
try:
    from scipy.stats import spearmanr
    sp = spearmanr(df["cnn_prob"], df["dock_affinity_kcalmol"])
    print(f"[METRIC] Spearman(cnn_prob, docking): {sp.correlation:.3f} (p={sp.pvalue:.2e})")
except Exception:
    pass

# Final ranked list (combine prob + docking)
ranked = df.sort_values(["dock_affinity_kcalmol","cnn_prob"], ascending=[True, False]).reset_index(drop=True)
ranked.to_csv(os.path.join(RUN_DIR,"ranked_results.csv"), index=False)


In [ ]:
# # ==== Step 7: Classical ML reranker for binding affinity ====
# # Uses scikit-learn; no quantum libs needed.

# import os, numpy as np, pandas as pd
# from pathlib import Path
# from rdkit import Chem
# from rdkit.Chem import AllChem, DataStructs, rdMolDescriptors, Descriptors
# from rdkit.Chem.Scaffolds import MurckoScaffold

# from sklearn.model_selection import GroupKFold, GridSearchCV
# from sklearn.metrics import mean_absolute_error, r2_score
# from sklearn.preprocessing import StandardScaler
# from sklearn.pipeline import Pipeline
# from sklearn.linear_model import Ridge
# from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
# from sklearn.decomposition import PCA
# from scipy.stats import spearmanr

# RUN_DIR = r"C:\Users\Pawan\Desktop\quantum based rug discovery\runs\vs1"
# CSV_IN  = os.path.join(RUN_DIR, "docking_results.csv")   # produced in Step 6
# CSV_OUT = os.path.join(RUN_DIR, "reranked_ml_results.csv")

# # --------- Feature builders ---------
# def smi_to_morgan_bits(smiles, n_bits=2048, radius=2):
#     m = Chem.MolFromSmiles(smiles)
#     if m is None: return None
#     arr = np.zeros((n_bits,), dtype=np.int8)
#     fp  = AllChem.GetMorganFingerprintAsBitVect(m, radius, nBits=n_bits)
#     DataStructs.ConvertToNumpyArray(fp, arr)
#     return arr

# def basic_physchem(mol):
#     # small, robust set; add/remove as you like
#     return [
#         Descriptors.MolWt(mol),
#         Descriptors.MolLogP(mol),
#         Descriptors.NumHAcceptors(mol),
#         Descriptors.NumHDonors(mol),
#         Descriptors.TPSA(mol),
#         Descriptors.NumRotatableBonds(mol),
#         rdMolDescriptors.CalcNumAromaticRings(mol),
#         rdMolDescriptors.CalcFractionCSP3(mol),
#     ]

# def smiles_to_features(smiles, use_physchem=True, bits=2048, radius=2):
#     m = Chem.MolFromSmiles(smiles)
#     if m is None: return None
#     # Morgan bits
#     arr = np.zeros((bits,), dtype=np.int8)
#     fp  = AllChem.GetMorganFingerprintAsBitVect(m, radius, nBits=bits)
#     DataStructs.ConvertToNumpyArray(fp, arr)
#     feats = arr.astype(np.float32)
#     # Optional physchem block
#     if use_physchem:
#         pc = np.array(basic_physchem(m), dtype=np.float32)
#         feats = np.concatenate([feats, pc], axis=0)
#     return feats

# def murcko_scaffold(smiles):
#     m = Chem.MolFromSmiles(smiles)
#     if m is None: return "NA"
#     smi = Chem.MolToSmiles(MurckoScaffold.GetScaffoldForMol(m), isomericSmiles=True)
#     return smi or "NA"

# # --------- Load docked set & choose target ---------
# df = pd.read_csv(CSV_IN).dropna(subset=["smiles", "dock_affinity_kcalmol"]).copy()

# # OPTION A (default): predict docking affinity (kcal/mol; more negative = better)
# target_name = "dock_affinity_kcalmol"
# y = df[target_name].values.astype(np.float32)

# # OPTION B (experimental): if you have standard_value (nM) for each row, use pActivity instead:
# # def to_pX(nm): 
# #     return None if pd.isna(nm) or nm <= 0 else (9 - np.log10(float(nm)))  # pIC50/pKi ~ -log10(M); nM → 10^-9 M
# # if "standard_value" in df.columns:
# #     y_alt = df["standard_value"].apply(to_pX)
# #     if y_alt.notna().sum() > int(0.6*len(df)):  # enough data
# #         df = df[y_alt.notna()].copy()
# #         y  = y_alt[y_alt.notna()].values.astype(np.float32)
# #         target_name = "pActivity"

# # --------- Build X and groups (scaffold split) ---------
# X_list, keep_idx, groups = [], [], []
# for i, s in enumerate(df["smiles"]):
#     v = smiles_to_features(s, use_physchem=True, bits=2048, radius=2)
#     if v is not None:
#         X_list.append(v)
#         keep_idx.append(i)
#         groups.append(murcko_scaffold(s))
# X = np.vstack(X_list).astype(np.float32)
# y = y[keep_idx]
# df = df.iloc[keep_idx].reset_index(drop=True)
# groups = np.array(groups)

# print(f"[INFO] Features: X={X.shape}, target={target_name}, N={len(y)}")

# # --------- Models & hyperparams ---------
# models = {
#     "ridge": Pipeline([
#         ("scaler", StandardScaler(with_mean=False)),  # sparse-ish; keep with_mean=False
#         ("pca", PCA(n_components=128, random_state=42)),
#         ("reg", Ridge(random_state=42))
#     ]),
#     "rf": Pipeline([
#         ("reg", RandomForestRegressor(
#             n_estimators=400, random_state=42, n_jobs=-1, max_features="sqrt"))
#     ]),
#     "gbr": Pipeline([
#         ("reg", GradientBoostingRegressor(random_state=42))
#     ]),
# }

# param_grid = {
#     "ridge": {
#         "pca__n_components": [64, 128, 256],
#         "reg__alpha": [0.1, 1.0, 5.0, 10.0],
#     },
#     "rf": {
#         "reg__n_estimators": [300, 600],
#         "reg__max_depth": [None, 12, 20],
#         "reg__min_samples_leaf": [1, 3],
#     },
#     "gbr": {
#         "reg__n_estimators": [300, 600],
#         "reg__learning_rate": [0.05, 0.1, 0.2],
#         "reg__max_depth": [2, 3, 4],
#     },
# }

# # --------- Scaffold CV (GroupKFold) ---------
# gkf = GroupKFold(n_splits=5)
# best_name, best_est, best_mae = None, None, 1e9

# for name, pipe in models.items():
#     grid = GridSearchCV(
#         pipe, param_grid[name], scoring="neg_mean_absolute_error",
#         cv=gkf.split(X, y, groups), n_jobs=-1, verbose=0
#     )
#     grid.fit(X, y)
#     mae = -grid.best_score_
#     print(f"[CV] {name}: MAE={mae:.3f} | best_params={grid.best_params_}")
#     if mae < best_mae:
#         best_mae, best_est, best_name = mae, grid.best_estimator_, name

# print(f"[SELECT] Best model: {best_name} (CV MAE={best_mae:.3f})")

# # --------- Final fit on all data, predict & evaluate hold-in metrics ---------
# best_est.fit(X, y)
# y_pred = best_est.predict(X)

# mae_in  = mean_absolute_error(y, y_pred)
# rmse_in = np.sqrt(np.mean((y - y_pred)**2))
# r2_in   = r2_score(y, y_pred)
# sp_in   = spearmanr(y, y_pred).correlation
# print(f"[FIT-ALL] MAE={mae_in:.2f}, RMSE={rmse_in:.2f}, R^2={r2_in:.2f}, Spearman={sp_in:.2f}")

# # --------- Rerank docking with ML for near-ties ---------
# df["ml_pred"] = y_pred

# def rerank(df, tie_window=0.2):
#     # Sort by docking (ascending; more negative is better), then within near-ties, by ml_pred (ascending)
#     df = df.sort_values("dock_affinity_kcalmol", ascending=True).reset_index(drop=True)
#     out = []
#     i = 0
#     while i < len(df):
#         j = i + 1
#         while j < len(df) and abs(df.loc[j,"dock_affinity_kcalmol"] - df.loc[i,"dock_affinity_kcalmol"]) <= tie_window:
#             j += 1
#         block = df.iloc[i:j].sort_values("ml_pred", ascending=True)
#         out.append(block)
#         i = j
#     return pd.concat(out, ignore_index=True)

# reranked = rerank(df, tie_window=0.2)
# reranked.to_csv(CSV_OUT, index=False)
# print("Saved:", CSV_OUT)


In [ ]:
# =================================================================================
# Modified Molecular Docking Script (for debugging)
# =================================================================================
import os, shutil, subprocess
import pandas as pd
from pathlib import Path
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs
import numpy as np

RUN_DIR = r"C:\Users\Pawan\Desktop\quantum based rug discovery\runs\vs1"
RECEPTOR = r"C:\Users\Pawan\Desktop\quantum based rug discovery\receptor.pdb"
POCKET = r"C:\Users\Pawan\Desktop\quantum based rug discovery\pocket.pdb"

# Check for executables
obabel = shutil.which("obabel")
if not obabel:
    raise RuntimeError("OpenBabel (obabel) not found on PATH.")
vina = shutil.which("vina")
if not vina:
    raise RuntimeError("Vina not found on PATH.")

def ensure_dir(d): Path(d).mkdir(parents=True, exist_ok=True)
ensure_dir(RUN_DIR)
prep_dir = os.path.join(RUN_DIR, "ligands_prep")
ensure_dir(prep_dir)
pose_dir = os.path.join(RUN_DIR, "poses")
ensure_dir(pose_dir)

# Read the screened ligands
screened = pd.read_csv(os.path.join(RUN_DIR, "screened_ligands.csv"))

# Receptor to PDBQT
receptor_pdbqt = os.path.join(RUN_DIR, "receptor.pdbqt")
subprocess.run([obabel, RECEPTOR, "-O", receptor_pdbqt, "-h", "--partialcharge", "gasteiger"], check=True)

# Function to get pocket center
def pdb_center_from_atoms(pdb_path):
    xs, ys, zs = [], [], []
    with open(pdb_path) as f:
        for line in f:
            if line.startswith(("ATOM", "HETATM")):
                try:
                    xs.append(float(line[30:38]))
                    ys.append(float(line[38:46]))
                    zs.append(float(line[46:54]))
                except ValueError:
                    continue
    if not xs:
        raise ValueError("No atom coords found for pocket.")
    return (sum(xs) / len(xs), sum(ys) / len(ys), sum(zs) / len(zs))

center = pdb_center_from_atoms(POCKET)
box = (20.0, 20.0, 20.0)
print("Dock center:", center, "box:", box)

# SMILES to PDBQT conversion using RDKit and OpenBabel
def smiles_to_sdf(smiles, out_sdf, seed=42):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return False
    mol = Chem.AddHs(mol)
    params = AllChem.ETKDGv3()
    params.randomSeed = seed
    if AllChem.EmbedMolecule(mol, params) != 0:
        return False
    try:
        AllChem.UFFOptimizeMolecule(mol, maxIters=500)
    except:
        pass
    w = Chem.SDWriter(out_sdf)
    w.write(mol)
    w.close()
    return True

prep_records = []
for _, r in screened.iterrows():
    name, smi = r["name"], r["smiles"]
    sdf = os.path.join(prep_dir, f"{name}.sdf")
    pdbqt = os.path.join(prep_dir, f"{name}.pdbqt")
    ok = smiles_to_sdf(smi, sdf)
    if not ok:
        prep_records.append((name, False, "SDF gen failed"))
        continue
    res = subprocess.run([obabel, sdf, "-O", pdbqt, "-h", "--partialcharge", "gasteiger"],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    prep_records.append((name, os.path.exists(pdbqt), "ok" if os.path.exists(pdbqt) else "obabel fail"))

pd.DataFrame(prep_records, columns=["name", "ok", "note"]).to_csv(os.path.join(RUN_DIR, "prep_status.csv"), index=False)
print("Prep done.")

# Docking with Vina
def parse_vina_best(log_text):
    for line in (log_text or "").splitlines():
        parts = line.strip().split()
        if len(parts) >= 2 and parts[0].isdigit():
            try:
                return float(parts[1])
            except ValueError:
                pass
    return None

rows = []
for _, r in screened.iterrows():
    name = r["name"]
    lig_pdbqt = os.path.join(RUN_DIR, "ligands_prep", f"{name}.pdbqt")
    out_pose = os.path.join(pose_dir, f"{name}_pose.pdbqt")
    if not os.path.exists(lig_pdbqt):
        continue
    
    # --- DEBUGGING PRINTS ADDED HERE ---
    print(f"\n--- Running Vina for {name} ---")
    print(f"Command: {vina} --receptor {os.path.join(RUN_DIR,'receptor.pdbqt')} --ligand {lig_pdbqt} ...")

    cmd = [
        vina,
        "--receptor", os.path.join(RUN_DIR, "receptor.pdbqt"),
        "--ligand", lig_pdbqt,
        "--center_x", str(center[0]), "--center_y", str(center[1]), "--center_z", str(center[2]),
        "--size_x", str(box[0]), "--size_y", str(box[1]), "--size_z", str(box[2]),
        "--exhaustiveness", "16",
        "--out", out_pose
    ]
    p = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    
    print("\n>>> Vina Raw Output <<<")
    print(p.stdout)
    print(">>> End of Vina Output <<<\n")
    
    aff = parse_vina_best(p.stdout)
    rows.append({
        "name": name,
        "smiles": r["smiles"],
        "label": int(r["label"]), # Assuming the label column exists in screened_ligands.csv
        "cnn_prob": float(r["cnn_prob"]),
        "dock_affinity_kcalmol": aff,
        "pose_path": out_pose if os.path.exists(out_pose) else ""
    })

res = pd.DataFrame(rows).dropna(subset=["dock_affinity_kcalmol"])
res.to_csv(os.path.join(RUN_DIR, "docking_results.csv"), index=False)
print("Docking done:", len(res), "poses")

In [ ]:
receptor_pdbqt = r"C:\Users\Pawan\Desktop\quantum based rug discovery\runs\vs1\receptor.pdbqt"

with open(receptor_pdbqt, "r", errors="replace") as f:
    lines = f.readlines()
for i in range(max(0, 1322-5), min(len(lines), 1322+5)):
    print(f"{i+1:>6}: {lines[i].rstrip()}")


In [ ]:
import sys, subprocess, shutil
from pathlib import Path

def quick_clean_pdb(src, dst):
    """Remove waters (HOH) and most HETATM except common ions; keep altloc ' ' or 'A'."""
    keep_ions = {"NA","CL","MG","MN","CA","ZN","K","FE","CU","CO","NI"}
    with open(src, "r", errors="replace") as f, open(dst, "w") as g:
        for line in f:
            if not line.startswith(("ATOM", "HETATM", "TER", "END")):
                continue
            if line.startswith("HETATM"):
                res = line[17:20].strip()
                if res == "HOH":
                    continue
                if res not in keep_ions:
                    continue
            alt = line[16]  # alternate location indicator
            if alt not in (" ", "A"):
                continue
            g.write(line)

def prepare_receptor_meeko(in_pdb: str, out_pdbqt: str,
                           allow_bad_res=True,
                           default_altloc="A",
                           use_prody=False):
    """
    Calls the installed mk_prepare_receptor (your build).
    Flags available per your --help output.
    """
    exe = shutil.which("mk_prepare_receptor.py") or shutil.which("mk_prepare_receptor")
    if not exe:
        raise RuntimeError("mk_prepare_receptor not found on PATH.")

    in_pdb = str(Path(in_pdb))
    out_pdbqt = str(Path(out_pdbqt))
    Path(out_pdbqt).parent.mkdir(parents=True, exist_ok=True)

    # Choose reader flag based on your help:
    # --read_pdb (simple) OR -i/--read_with_prody (more tolerant, needs ProDy installed)
    if use_prody:
        cmd = [exe, "-i", in_pdb, "--write_pdbqt", out_pdbqt]
    else:
        cmd = [exe, "--read_pdb", in_pdb, "--write_pdbqt", out_pdbqt]

    # Optional flags that DO exist in your help:
    if allow_bad_res:
        cmd += ["-a"]  # --allow_bad_res
    if default_altloc:
        cmd += ["--default_altloc", str(default_altloc)]

    print("Running:", " ".join(cmd))
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        print("\n--- ERROR (stderr) ---")
        print(proc.stderr.strip() or "<no stderr>")
        print("\n--- OUTPUT (stdout) ---")
        print(proc.stdout.strip() or "<no stdout>")
        raise RuntimeError(f"mk_prepare_receptor failed with exit code {proc.returncode}")
    else:
        print("Success ->", out_pdbqt)

# --- Use it: pre-clean then prepare ---
raw_pdb = r"C:\Users\Pawan\Desktop\quantum based rug discovery\receptor.pdb"
clean_pdb = r"C:\Users\Pawan\Desktop\quantum based rug discovery\receptor_clean.pdb"
out_pdbqt = r"C:\Users\Pawan\Desktop\quantum based rug discovery\runs\vs1\receptor.pdbqt"

quick_clean_pdb(raw_pdb, clean_pdb)
prepare_receptor_meeko(clean_pdb, out_pdbqt, allow_bad_res=True, default_altloc="A", use_prody=False)


In [ ]:
def validate_receptor_pdbqt(path):
    illegal = ("ROOT","BRANCH","ENDBRANCH","TORSDOF","ENDROOT","MODEL","ENDMDL","CONECT","MASTER","USER")
    bad=[]
    with open(path, "r", errors="replace") as f:
        for i, line in enumerate(f, 1):
            if line.lstrip().startswith(illegal):
                bad.append((i, line.rstrip()))
    return bad

issues = validate_receptor_pdbqt(r"C:\Users\Pawan\Desktop\quantum based rug discovery\runs\vs1\receptor.pdbqt")
print("Illegal tags:", issues[:5] if issues else "none")


In [ ]:
# =================================================================================
# Molecular Docking Pipeline (Meeko receptor + RDKit/OpenBabel ligands + Vina)
# - Keeps the Meeko-made receptor.pdbqt; never overwrites it.
# - Uses pocket center from a PDB pocket file.
# - Deterministic RDKit 3D (seeded ETKDG + UFF).
# - Clear logging + robust CSV outputs.
# =================================================================================

import os, sys, shutil, subprocess
from pathlib import Path
import pandas as pd
import numpy as np

# RDKit imports
from rdkit import Chem
from rdkit.Chem import AllChem

# -----------------------------
# User paths
# -----------------------------
RUN_DIR   = r"C:\Users\Pawan\Desktop\quantum based rug discovery\runs\vs1"
RECEPTOR  = r"C:\Users\Pawan\Desktop\quantum based rug discovery\receptor.pdb"         # used only if we need to (re)make PDBQT with Meeko
POCKET    = r"C:\Users\Pawan\Desktop\quantum based rug discovery\pocket.pdb"          # pocket PDB (to compute center)
CSV_IN    = os.path.join(RUN_DIR, "screened_ligands.csv")                             # must have: name, smiles, label, cnn_prob
RECEPTOR_PDBQT = os.path.join(RUN_DIR, "receptor.pdbqt")                              # expected Meeko output

# ligand prep/output dirs
LIG_PREP_DIR = os.path.join(RUN_DIR, "ligands_prep")
POSE_DIR     = os.path.join(RUN_DIR, "poses")

# Dock box (Å)
BOX_SIZE = (20.0, 20.0, 20.0)

# -----------------------------
# Executables
# -----------------------------
def which_any(*names):
    for n in names:
        p = shutil.which(n)
        if p:
            return p
    return None

OBABEL = which_any("obabel")
if not OBABEL:
    raise RuntimeError("OpenBabel 'obabel' not found on PATH.")

# Prefer AutoDock Vina, fall back to QuickVina 2 if available under common names
VINA = which_any("vina", "qvina2", "quickvina2")
if not VINA:
    raise RuntimeError("Vina/QuickVina executable not found on PATH (tried: vina, qvina2, quickvina2).")

# Meeko (optional auto-prepare if receptor.pdbqt missing)
MEEKO_PREP = which_any("mk_prepare_receptor.py", "mk_prepare_receptor")

# -----------------------------
# Utils
# -----------------------------
def ensure_dir(p: str):
    Path(p).mkdir(parents=True, exist_ok=True)

def log(*args):
    print(*args, flush=True)

def file_exists(p: str) -> bool:
    try:
        return Path(p).exists()
    except Exception:
        return False

# -----------------------------
# Pocket center (simple average)
# -----------------------------
def pdb_center_from_atoms(pdb_path: str):
    xs, ys, zs = [], [], []
    with open(pdb_path, "r", errors="replace") as f:
        for line in f:
            if line.startswith(("ATOM", "HETATM")):
                try:
                    xs.append(float(line[30:38]))
                    ys.append(float(line[38:46]))
                    zs.append(float(line[46:54]))
                except ValueError:
                    continue
    if not xs:
        raise ValueError(f"No atom coords found in {pdb_path}")
    return (sum(xs) / len(xs), sum(ys) / len(ys), sum(zs) / len(zs))

# -----------------------------
# RDKit: SMILES -> SDF (seeded/deterministic)
# -----------------------------
def smiles_to_sdf(smiles: str, out_sdf: str, seed: int = 42) -> bool:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return False
    mol = Chem.AddHs(mol)
    params = AllChem.ETKDGv3()
    params.randomSeed = seed
    if AllChem.EmbedMolecule(mol, params) != 0:
        return False
    try:
        AllChem.UFFOptimizeMolecule(mol, maxIters=500)
    except Exception:
        # ignore failures; keep embedded geometry
        pass
    w = Chem.SDWriter(out_sdf)
    w.write(mol)
    w.close()
    return True

# -----------------------------
# OPTIONAL: Meeko receptor helpers
# -----------------------------
def quick_clean_pdb(src: str, dst: str):
    """Remove waters (HOH) and most HETATM except common ions; keep altloc ' ' or 'A'."""
    keep_ions = {"NA","CL","MG","MN","CA","ZN","K","FE","CU","CO","NI"}
    with open(src, "r", errors="replace") as f, open(dst, "w") as g:
        for line in f:
            if not line.startswith(("ATOM", "HETATM", "TER", "END")):
                continue
            if line.startswith("HETATM"):
                res = line[17:20].strip()
                if res == "HOH":
                    continue
                if res not in keep_ions:
                    continue
            alt = line[16]  # alternate location indicator
            if alt not in (" ", "A"):
                continue
            g.write(line)

def prepare_receptor_meeko(in_pdb: str, out_pdbqt: str,
                           allow_bad_res: bool = True,
                           default_altloc: str = "A",
                           use_prody: bool = False):
    """
    Calls mk_prepare_receptor if available. Uses flags that exist in its --help.
    """
    if not MEEKO_PREP:
        raise RuntimeError("mk_prepare_receptor not found on PATH.")
    in_pdb = str(Path(in_pdb))
    out_pdbqt = str(Path(out_pdbqt))
    Path(out_pdbqt).parent.mkdir(parents=True, exist_ok=True)

    if use_prody:
        cmd = [MEEKO_PREP, "-i", in_pdb, "--write_pdbqt", out_pdbqt]
    else:
        cmd = [MEEKO_PREP, "--read_pdb", in_pdb, "--write_pdbqt", out_pdbqt]

    if allow_bad_res:
        cmd += ["-a"]  # --allow_bad_res
    if default_altloc:
        cmd += ["--default_altloc", str(default_altloc)]

    log("Running Meeko:", " ".join(cmd))
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        log("\n--- Meeko ERROR (stderr) ---\n", proc.stderr.strip() or "<no stderr>")
        log("\n--- Meeko OUTPUT (stdout) ---\n", proc.stdout.strip() or "<no stdout>")
        raise RuntimeError(f"mk_prepare_receptor failed (exit {proc.returncode})")
    log("Meeko receptor created:", out_pdbqt)

def validate_receptor_pdbqt_for_vina(path: str):
    """
    Check for records that commonly upset Vina/QuickVina receptors.
    We'll just warn; Meeko output should already be fine.
    """
    illegal = ("ROOT","BRANCH","ENDBRANCH","TORSDOF","ENDROOT","MODEL","ENDMDL","CONECT","MASTER","USER")
    bad = []
    with open(path, "r", errors="replace") as f:
        for i, line in enumerate(f, 1):
            if line.lstrip().startswith(illegal):
                bad.append((i, line.rstrip()))
    return bad

def peek_lines(path: str, lineno: int, radius: int = 6):
    with open(path, "r", errors="replace") as f:
        lines = f.readlines()
    a = max(1, lineno - radius)
    b = min(len(lines), lineno + radius)
    for i in range(a, b+1):
        print(f"{i:6}: {lines[i-1].rstrip()}")

# -----------------------------
# Vina table parser (best affinity = row 1, col 2)
# -----------------------------
def parse_vina_best(log_text: str):
    """
    Finds the first docking mode line:
        1   -8.3   0.000   0.000 ...
    Returns float(kcal/mol) or None.
    """
    if not log_text:
        return None
    for line in log_text.splitlines():
        s = line.strip()
        parts = s.split()
        if len(parts) >= 2 and parts[0].isdigit():
            try:
                return float(parts[1])
            except ValueError:
                continue
    return None

# -----------------------------
# Main
# -----------------------------
def main():
    # Create dirs
    ensure_dir(RUN_DIR)
    ensure_dir(LIG_PREP_DIR)
    ensure_dir(POSE_DIR)

    # Read screened ligands
    if not file_exists(CSV_IN):
        raise FileNotFoundError(f"Missing ligand CSV: {CSV_IN}")
    screened = pd.read_csv(CSV_IN)
    required_cols = {"name", "smiles"}
    if not required_cols.issubset(screened.columns):
        raise ValueError(f"{CSV_IN} must contain columns: {sorted(required_cols)}")
    # Optional columns; fill defaults if absent
    if "label" not in screened.columns:
        screened["label"] = 0
    if "cnn_prob" not in screened.columns:
        screened["cnn_prob"] = np.nan

    # Receptor: must be Meeko-made PDBQT already. Do not overwrite.
    if not file_exists(RECEPTOR_PDBQT):
        # Try to create it via Meeko, if available; otherwise, instruct user
        if not file_exists(RECEPTOR):
            raise FileNotFoundError(
                f"Expected Meeko receptor at {RECEPTOR_PDBQT}, and raw PDB not found at {RECEPTOR}.\n"
                f"Please supply receptor.pdbqt produced by mk_prepare_receptor."
            )
        if not MEEKO_PREP:
            raise RuntimeError(
                f"Expected Meeko receptor at {RECEPTOR_PDBQT} but mk_prepare_receptor is not on PATH.\n"
                f"Install meeko or run receptor preparation separately."
            )
        # Optional: pre-clean PDB then prepare
        clean_pdb = os.path.join(Path(RECEPTOR).parent, "receptor_clean.pdb")
        log("Pre-cleaning receptor PDB (remove HOH, non-ion HETATMs, altloc != A/' '):", clean_pdb)
        quick_clean_pdb(RECEPTOR, clean_pdb)
        prepare_receptor_meeko(clean_pdb, RECEPTOR_PDBQT, allow_bad_res=True, default_altloc="A", use_prody=False)
    else:
        log("Using existing Meeko receptor:", RECEPTOR_PDBQT)

    # Sanity check receptor for tags that Vina dislikes (warn only)
    bad = validate_receptor_pdbqt_for_vina(RECEPTOR_PDBQT)
    if bad:
        log(f"WARNING: Found {len(bad)} suspicious lines in receptor.pdbqt (Vina may fail). Showing first 5:")
        for i, rec in enumerate(bad[:5], 1):
            log(f"  {i}) line {rec[0]}: {rec[1]}")
        log("Consider re-preparing receptor with Meeko.")

    # Compute docking center
    center = pdb_center_from_atoms(POCKET)
    box = BOX_SIZE
    log("Dock center:", center, "box:", box)
    log("Using Vina executable:", VINA)
    log("Using OpenBabel executable:", OBABEL)

    # -------------------------
    # Ligand prep (RDKit -> SDF -> OpenBabel -> PDBQT)
    # -------------------------
    prep_records = []
    for _, row in screened.iterrows():
        name = str(row["name"])
        smi  = str(row["smiles"])
        sdf  = os.path.join(LIG_PREP_DIR, f"{name}.sdf")
        lpdbqt = os.path.join(LIG_PREP_DIR, f"{name}.pdbqt")

        ok = smiles_to_sdf(smi, sdf, seed=42)
        if not ok:
            prep_records.append((name, False, "RDKit SDF generation failed"))
            continue

        # Convert to PDBQT with Gasteiger charges & hydrogens
        res = subprocess.run(
            [OBABEL, sdf, "-O", lpdbqt, "-h", "--partialcharge", "gasteiger"],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
        )
        success = file_exists(lpdbqt)
        note = "ok" if success else f"obabel fail: {res.stdout.strip()[:120]}..."
        prep_records.append((name, success, note))

    prep_df = pd.DataFrame(prep_records, columns=["name", "ok", "note"])
    prep_csv = os.path.join(RUN_DIR, "prep_status.csv")
    prep_df.to_csv(prep_csv, index=False)
    log(f"Prep done. Wrote: {prep_csv}  (ok={prep_df['ok'].sum()}/{len(prep_df)})")

    # -------------------------
    # Docking
    # -------------------------
    rows = []
    for _, row in screened.iterrows():
        name = str(row["name"])
        lig_pdbqt = os.path.join(LIG_PREP_DIR, f"{name}.pdbqt")
        out_pose  = os.path.join(POSE_DIR, f"{name}_pose.pdbqt")

        if not file_exists(lig_pdbqt):
            log(f"Skip {name}: ligand pdbqt not found.")
            continue

        log(f"\n--- Running Vina for {name} ---")
        cmd = [
            VINA,
            "--receptor", RECEPTOR_PDBQT,
            "--ligand", lig_pdbqt,
            "--center_x", str(center[0]), "--center_y", str(center[1]), "--center_z", str(center[2]),
            "--size_x", str(box[0]), "--size_y", str(box[1]), "--size_z", str(box[2]),
            "--exhaustiveness", "16",
            "--out", out_pose
        ]
        log("Command:", " ".join(cmd))
        p = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

        log("\n>>> Vina Raw Output <<<")
        log(p.stdout.rstrip("\n"))
        log(">>> End of Vina Output <<<\n")

        aff = parse_vina_best(p.stdout)
        rows.append({
            "name": name,
            "smiles": row.get("smiles", ""),
            "label": int(row.get("label", 0)),
            "cnn_prob": float(row.get("cnn_prob", np.nan)) if pd.notna(row.get("cnn_prob", np.nan)) else np.nan,
            "dock_affinity_kcalmol": aff,
            "pose_path": out_pose if file_exists(out_pose) else ""
        })

        # Detect parse error lines for quick triage
        if "Parse error on line" in p.stdout:
            # Attempt to extract line number and peek
            try:
                # Example: Parse error on line 1322 in file "path": Unknown or inappropriate tag
                for ln in p.stdout.splitlines():
                    if "Parse error on line" in ln:
                        seg = ln.split("Parse error on line", 1)[1].strip()
                        line_no = int(seg.split()[0])
                        log(f"Peek receptor lines near {line_no}:")
                        peek_lines(RECEPTOR_PDBQT, line_no, radius=6)
                        break
            except Exception:
                pass

    res = pd.DataFrame(rows)
    # keep only rows that have a parsed affinity
    res = res.dropna(subset=["dock_affinity_kcalmol"])
    out_csv = os.path.join(RUN_DIR, "docking_results.csv")
    res.to_csv(out_csv, index=False)
    log(f"Docking done: {len(res)} poses  -> {out_csv}")

if __name__ == "__main__":
    main()


In [ ]:
import os, pandas as pd, numpy as np
import matplotlib.pyplot as plt

RUN_DIR = r"C:\Users\Pawan\Desktop\quantum based rug discovery\runs\vs1"
csv_path = os.path.join(RUN_DIR, "docking_results.csv")

df = pd.read_csv(csv_path)

# Clean columns defensively
if 'dock_affinity_kcalmol' in df.columns:
    df['dock_affinity_kcalmol'] = pd.to_numeric(df['dock_affinity_kcalmol'], errors='coerce')

# Typical Vina convention: more negative = better. Positives are usually failures/clashes.
df['is_valid_aff'] = df['dock_affinity_kcalmol'].notna()
df_valid = df[df['is_valid_aff']].copy()

# Keep a "score" where higher is better for plotting (negate affinities)
df_valid['score'] = -df_valid['dock_affinity_kcalmol']

# Leaderboard (top 20 by most negative affinity)
top = df_valid.sort_values('dock_affinity_kcalmol').head(20)
print("Top 20 by affinity (kcal/mol, more negative is better):")
print(top[['name','dock_affinity_kcalmol','cnn_prob','label','pose_path']].fillna(''))

# Save just the leaderboard to a CSV for quick sharing
top.to_csv(os.path.join(RUN_DIR, "top20_docking.csv"), index=False)
print("\nSaved:", os.path.join(RUN_DIR, "top20_docking.csv"))


In [ ]:
vals = df_valid['dock_affinity_kcalmol'].dropna().values

plt.figure(figsize=(6,4))
plt.hist(vals, bins=30)
plt.xlabel("Affinity (kcal/mol)  [more negative is better]")
plt.ylabel("Count")
plt.title("Distribution of docking affinities")
plt.tight_layout()
plt.show()


In [ ]:
if 'cnn_prob' in df_valid.columns:
    sub = df_valid[df_valid['cnn_prob'].notna()].copy()
    if not sub.empty:
        # We’ll plot CNN prob (0–1) vs -affinity (so higher means better bind)
        plt.figure(figsize=(6,4))
        plt.scatter(sub['cnn_prob'].astype(float), sub['score'])
        plt.xlabel("cnn_prob")
        plt.ylabel("Binding score = -affinity (kcal/mol)")
        plt.title("cnn_prob vs docking score")
        plt.tight_layout()
        plt.show()

        # correlation
        corr = np.corrcoef(sub['cnn_prob'].astype(float), sub['score'])[0,1]
        print(f"Pearson corr(cnn_prob, -affinity) = {corr:.3f}")
    else:
        print("cnn_prob column present but empty after cleaning.")
else:
    print("cnn_prob column not found; skipping correlation plot.")


In [ ]:
pip install py3Dmol


In [ ]:
import py3Dmol

receptor_pdbqt = os.path.join(RUN_DIR, "receptor.pdbqt")
example_pose = top.iloc[0]['pose_path']  # or any path from the leaderboard
print("Viewing pose:", example_pose)

# read files
with open(receptor_pdbqt, 'r') as f: rec_txt = f.read()
with open(example_pose, 'r') as f: lig_txt = f.read()

# viewer
view = py3Dmol.view(width=700, height=500)
view.addModel(rec_txt, 'pdbqt')   # receptor
view.setStyle({'model':0}, {"cartoon":{}})  # cartoon for protein

view.addModel(lig_txt, 'pdbqt')   # ligand
view.setStyle({'model':1}, {"stick":{}})    # sticks for ligand
view.zoomTo()
view.show()


In [ ]:
view.setStyle({'model':0}, {"line":{}})


In [ ]:
df_valid = df_valid[df_valid['dock_affinity_kcalmol'] < 0]
